# 00 — Environment and config (Qwen3-0.6B)

Goal of this series: our own PyTorch Qwen3 whose activations match Hugging Face layer by layer, then profile it.
This notebook only prepares the ground: read the real config, count the parameters by hand, and decide what fits in 4 GB of GPU memory.

**What we do** — set the seed, the `FORCE_CPU` flag, and the paths; check the installed libraries.

**Why** — every later notebook starts from the same setup, and the series needs `transformers` (reference model, tokenizer) next to our own code.

**What you should see** — library versions and the device, then `PASS`.

In [1]:
import sys, pathlib, json, math
sys.path.insert(0, str(pathlib.Path().resolve().parent))   # qwen/
import numpy as np, torch, transformers, safetensors

SEED = 0
FORCE_CPU = False
torch.manual_seed(SEED); np.random.seed(SEED)

device = torch.device("cpu" if FORCE_CPU or not torch.cuda.is_available() else "cuda")
MODEL_DIR = pathlib.Path().resolve().parent / "artifacts" / "Qwen3-0.6B"

print("torch", torch.__version__, "| transformers", transformers.__version__, "| safetensors", safetensors.__version__)
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
assert (MODEL_DIR / "config.json").exists() and (MODEL_DIR / "model.safetensors").exists(), "FAIL: weights not downloaded"
print("PASS: libraries installed, weights present")

torch 2.6.0+cu124 | transformers 5.22.1 | safetensors 0.8.0
device: cuda NVIDIA GeForce RTX 2050
PASS: libraries installed, weights present


### Definitions of the config fields

Symbols are the ones used in the parameter-count cell and in later notebooks.

| Field (symbol) | Value | Definition | Where it is used |
|---|---|---|---|
| `num_hidden_layers` (L) | 28 | Number of stacked, identically shaped decoder blocks. Each block = attention + MLP, each with its own weights; the output of block l is the input of block l+1. | notebook 05, 06 |
| `hidden_size` (H) | 1024 | Length of the vector that represents one token between blocks (the residual stream). The embedding table, every norm, and the input of every projection use this width. | everywhere |
| `num_attention_heads` (nq) | 16 | Number of query heads. Each head computes its own attention scores over the sequence, using a query vector of length `head_dim`. | notebook 03 |
| `num_key_value_heads` (nkv) | 8 | Number of key heads and of value heads. Each K/V head is shared by nq / nkv = 2 query heads (grouped-query attention, GQA). Fewer K/V heads means smaller `k_proj`, `v_proj` and a smaller KV cache. | notebook 03, 07 |
| `head_dim` (hd) | 128 | Length of one q, k or v head vector. In Qwen3 it is set independently of H: the q vector has nq·hd = 2048 entries, larger than H, and `o_proj` maps 2048 → 1024. | notebook 03, 04 |
| `intermediate_size` (I) | 3072 | Width of the MLP hidden layer: `gate_proj` and `up_proj` map H → I, `down_proj` maps I → H. | notebook 02 |
| `vocab_size` (V) | 151936 | Number of rows of the embedding table = number of token ids the model can read, and the length of the logits vector at each position. It can be larger than the number of tokens the tokenizer defines (checked in notebook 01). | notebook 01, 06 |
| `rms_norm_eps` | 1e-6 | The ε in RMSNorm, `x / sqrt(mean(x²) + ε) · weight`; it keeps the denominator above zero. | notebook 02 |
| `rope_theta` | 1,000,000 | Base θ of rotary position embeddings (RoPE). Pair i of a head vector is rotated by an angle `position · θ^(−2i/hd)`; a larger θ makes the slow-rotating pairs rotate more slowly, which suits long contexts. | notebook 04 |
| `tie_word_embeddings` | True | The output head reuses the embedding table: `logits = h · Eᵀ`. The file still stores a second copy (`lm_head.weight`). | notebook 06 |
| `attention_bias` | False | The q, k, v, o projections have no bias term: `y = W·x` only. | notebook 03 |
| `hidden_act` | silu | Activation inside the MLP: `silu(x) = x · sigmoid(x)`. The MLP is SwiGLU: `down_proj( silu(gate_proj(x)) ⊙ up_proj(x) )`. | notebook 02 |
| `max_position_embeddings` | 40960 | Longest sequence the model is configured for (the positions RoPE is defined over). The usable length on this GPU is set by KV cache memory (memory cell below). | notebook 07 |
| `torch_dtype` / `dtype` | bfloat16 | Number format of the stored weights: 2 bytes per value, 8 exponent bits and 7 mantissa bits, so the same range as fp32 with lower precision. | notebook 06 |

Derived: queries per K/V head = nq / nkv = 16 / 8 = 2; q width = nq·hd = 2048; k and v width = nkv·hd = 1024.

**What we do** — read `config.json` and print the fields that define the architecture.

**Why** — the notes said "Qwen3-2B"; the config is the only authority on layer count, sizes and head layout.

**What you should see** — 28 layers, hidden size 1024, 16 query heads, 8 key/value heads, head_dim 128, intermediate size 3072, vocabulary 151936, tied embeddings.

In [2]:
cfg = json.load(open(MODEL_DIR / "config.json"))
keys = ["num_hidden_layers", "hidden_size", "num_attention_heads", "num_key_value_heads", "head_dim",
        "intermediate_size", "vocab_size", "rms_norm_eps", "rope_theta", "tie_word_embeddings",
        "attention_bias", "hidden_act", "max_position_embeddings", "torch_dtype"]
for k in keys:
    print(f"{k:26s} {cfg.get(k, cfg.get('dtype') if k == 'torch_dtype' else None)}")

assert cfg["num_hidden_layers"] == 28 and cfg["hidden_size"] == 1024, "FAIL: not the expected 0.6B shape"
assert cfg["num_attention_heads"] % cfg["num_key_value_heads"] == 0, "FAIL: GQA needs q heads divisible by kv heads"
print("PASS: config read; q heads per kv head =", cfg["num_attention_heads"] // cfg["num_key_value_heads"])

num_hidden_layers          28
hidden_size                1024
num_attention_heads        16
num_key_value_heads        8
head_dim                   128
intermediate_size          3072
vocab_size                 151936
rms_norm_eps               1e-06
rope_theta                 1000000
tie_word_embeddings        True
attention_bias             False
hidden_act                 silu
max_position_embeddings    40960
torch_dtype                bfloat16
PASS: config read; q heads per kv head = 2


**What we do** — count the parameters by hand, component by component, from the config only.

**Why** — if the hand count equals the number of elements stored in the file, we understand every tensor the model contains (the same shapes are mapped to code in notebook 01).

**What you should see** — one line per component per layer, a total of 596,049,920 unique parameters, and `PASS`.

In [3]:
H, L = cfg["hidden_size"], cfg["num_hidden_layers"]
nq, nkv, hd, I, V = cfg["num_attention_heads"], cfg["num_key_value_heads"], cfg["head_dim"], cfg["intermediate_size"], cfg["vocab_size"]

per_layer = {
    "q_proj  (H -> nq*hd)":     H * nq * hd,
    "k_proj  (H -> nkv*hd)":    H * nkv * hd,
    "v_proj  (H -> nkv*hd)":    H * nkv * hd,
    "o_proj  (nq*hd -> H)":     nq * hd * H,
    "q_norm  (hd)":             hd,
    "k_norm  (hd)":             hd,
    "gate_proj (H -> I)":       H * I,
    "up_proj   (H -> I)":       H * I,
    "down_proj (I -> H)":       I * H,
    "input_layernorm (H)":      H,
    "post_attention_layernorm (H)": H,
}
for name, n in per_layer.items():
    print(f"{name:32s} {n:>12,}")
layer_total = sum(per_layer.values())
embed, final_norm = V * H, H
total = embed + L * layer_total + final_norm
print(f"{'one layer':32s} {layer_total:>12,}\n{'embedding (tied head)':32s} {embed:>12,}\n{'total':32s} {total:>12,}")

assert total == 596_049_920, "FAIL: hand count differs from 596,049,920"
print("PASS: hand-counted parameters = 596,049,920")

q_proj  (H -> nq*hd)                2,097,152
k_proj  (H -> nkv*hd)               1,048,576
v_proj  (H -> nkv*hd)               1,048,576
o_proj  (nq*hd -> H)                2,097,152
q_norm  (hd)                              128
k_norm  (hd)                              128
gate_proj (H -> I)                  3,145,728
up_proj   (H -> I)                  3,145,728
down_proj (I -> H)                  3,145,728
input_layernorm (H)                     1,024
post_attention_layernorm (H)            1,024
one layer                          15,730,944
embedding (tied head)             155,582,464
total                             596,049,920
PASS: hand-counted parameters = 596,049,920


**What we do** — open the safetensors file lazily (shapes only) and sum the elements of every tensor; compare `lm_head.weight` with `model.embed_tokens.weight`.

**Why** — the file has 751,632,384 elements, more than our count. The config says the head is tied to the embedding; the file stores the head tensor anyway.

**What you should see** — 311 tensors, 751,632,384 elements; the difference equals vocab × hidden; the two tensors are bit-identical; `PASS`.

In [4]:
from safetensors import safe_open
f = safe_open(str(MODEL_DIR / "model.safetensors"), "pt")
names = list(f.keys())
file_total = sum(math.prod(f.get_slice(n).get_shape()) for n in names)
same = torch.equal(f.get_tensor("lm_head.weight"), f.get_tensor("model.embed_tokens.weight"))
print(len(names), "tensors,", f"{file_total:,}", "elements in the file")
print("file - hand count =", f"{file_total - total:,}", "| vocab*hidden =", f"{V * H:,}")
print("lm_head.weight identical to embed_tokens.weight:", same)

assert file_total - total == V * H, "FAIL: unexplained extra elements in the file"
assert same, "FAIL: lm_head is not the embedding table"
print("PASS: the only extra tensor is the tied head, a copy of the embedding table")

311 tensors, 751,632,384 elements in the file
file - hand count = 155,582,464 | vocab*hidden = 155,582,464
lm_head.weight identical to embed_tokens.weight: True
PASS: the only extra tensor is the tied head, a copy of the embedding table


**What we do** — compute the memory of the weights in fp32 and bf16, and the KV cache per token and for several context lengths.

**Why** — the GPU has 4 GB. The weights must leave room for the KV cache and for activations; this decides when we can use fp32 (tight comparison with Hugging Face) and when bf16.

**What you should see** — weights 2.4 GB (fp32) / 1.2 GB (bf16), KV cache 112 KiB per token, and the context length at which the cache would reach 1 GiB.

In [5]:
free_b, total_b = torch.cuda.mem_get_info() if device.type == "cuda" else (0, 0)
GiB = 1024 ** 3
w_fp32, w_bf16 = total * 4, total * 2
kv_per_token = L * 2 * nkv * hd * 2          # layers x (K and V) x kv heads x head_dim x 2 bytes (bf16)
print(f"weights fp32 {w_fp32 / GiB:.2f} GiB | bf16 {w_bf16 / GiB:.2f} GiB | GPU free {free_b / GiB:.2f} GiB of {total_b / GiB:.2f}")
print(f"KV cache per token: {kv_per_token:,} B = {kv_per_token / 1024:.0f} KiB")
for n in (512, 4096, 32768):
    print(f"  {n:>6} tokens -> {n * kv_per_token / GiB:.3f} GiB")
print("context length for 1 GiB of KV cache:", GiB // kv_per_token, "tokens")

assert kv_per_token == 114_688, "FAIL: KV cache size per token"
if device.type == "cuda":
    assert w_fp32 < free_b, "FAIL: fp32 weights do not fit"
print("PASS: memory budget computed; fp32 weights fit the GPU")

weights fp32 2.22 GiB | bf16 1.11 GiB | GPU free 3.22 GiB of 4.00
KV cache per token: 114,688 B = 112 KiB
     512 tokens -> 0.055 GiB
    4096 tokens -> 0.438 GiB
   32768 tokens -> 3.500 GiB
context length for 1 GiB of KV cache: 9362 tokens
PASS: memory budget computed; fp32 weights fit the GPU


**What we do** — load the Hugging Face reference model in fp32 on the device, run one forward pass, and read the peak GPU memory; then release it.

**Why** — the reference must run on this machine, next to our implementation, to produce the activations we compare against. This also checks the real memory cost against the budget above.

**What you should see** — logits of shape (1, 5, 151936), peak memory slightly above the 2.2 GiB weight size, and `PASS`.

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
ref = AutoModelForCausalLM.from_pretrained(MODEL_DIR, dtype=torch.float32).to(device).eval()
ids = tok("The capital of France is", return_tensors="pt").input_ids.to(device)
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    logits = ref(ids).logits
peak = torch.cuda.max_memory_allocated() / GiB if device.type == "cuda" else float("nan")
print("tokens:", ids.tolist()[0], "| logits", tuple(logits.shape), f"| peak GPU {peak:.2f} GiB")
print("next token:", repr(tok.decode(logits[0, -1].argmax())))

assert logits.shape == (1, ids.shape[1], V), "FAIL: logits shape"
assert sum(p.numel() for p in ref.parameters()) == total, "FAIL: HF parameter count differs from the hand count"
print("PASS: Hugging Face reference runs; its parameter count equals the hand count")
del ref, logits
if device.type == "cuda":
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

tokens: [785, 6722, 315, 9625, 374] | logits (1, 5, 151936) | peak GPU 2.23 GiB
next token: ' Paris'
PASS: Hugging Face reference runs; its parameter count equals the hand count
